# Credit Card Fraud Detection

## Exploratory Data Analysis

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

DATA_PATH = 'train.csv'
df = pd.read_csv(DATA_PATH)

print(f'Kích thước tập dữ liệu: {df.shape[0]:,} dòng x {df.shape[1]} cột')
display(df.head())

Class Imbalance (`is_fraud`), Missing Values, and Duplicate `id` Check

In [ ]:
print('=' * 70)
print('1. KIỂM TRA MẤT CÂN BẰNG NHÃN, MISSING VALUES VÀ DUPLICATE ID')
print('=' * 70)

# 1.1 Thống kê nhãn is_fraud
fraud_counts = df['is_fraud'].value_counts().sort_index()
fraud_pct = df['is_fraud'].value_counts(normalize=True).sort_index() * 100
imbalance_df = pd.DataFrame({
    'Nhãn': ['0 (Hợp lệ - Non-Fraud)', '1 (Gian lận - Fraud)'],
    'Số lượng': fraud_counts.values,
    'Tỷ lệ (%)': fraud_pct.values.round(4)
})
display(imbalance_df)

if 1 in fraud_counts and fraud_counts[1] > 0:
    ratio = fraud_counts[0] / fraud_counts[1]
    print(f'-> Tỷ lệ mất cân bằng (Non-Fraud : Fraud) = {ratio:.2f} : 1\n')

# 1.2 Kiểm tra Duplicate ID & dòng trùng lặp hoàn toàn
dup_id_count = df['id'].duplicated().sum()
dup_row_count = df.drop(columns=['id'], errors='ignore').duplicated().sum()
print(f'- Số lượng ID bị trùng lặp (duplicate id): {dup_id_count:,}')
print(f'- Số lượng bản ghi trùng lặp thuộc tính (không tính id): {dup_row_count:,}\n')

# 1.3 Kiểm tra Missing Values trên toàn bộ 26 cột
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df)) * 100
missing_df = pd.DataFrame({
    'Số lượng Missing': missing_count,
    'Tỷ lệ Missing (%)': missing_pct.round(4),
    'Kiểu dữ liệu': df.dtypes
}).sort_values(by='Số lượng Missing', ascending=False)

if missing_df['Số lượng Missing'].sum() == 0:
    print('-> Tập dữ liệu KHÔNG có giá trị khuyết thiếu (0 missing values).')
else:
    print('-> Các cột có giá trị khuyết thiếu:')
    display(missing_df[missing_df['Số lượng Missing'] > 0])

# 1.4 Trực quan hóa tỷ lệ nhãn is_fraud
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ['#2b5c8f', '#d9534f']
bars = axes[0].bar(['Hợp lệ (0)', 'Gian lận (1)'], fraud_counts.values, color=colors, edgecolor='black')
axes[0].set_title('Số lượng giao dịch theo nhãn is_fraud', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Số lượng giao dịch')
for bar, pct in zip(bars, fraud_pct.values):
    yval = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width() / 2, yval, f'{yval:,}\n({pct:.2f}%)',
                 ha='center', va='bottom', fontweight='bold')
axes[0].set_ylim(0, fraud_counts.max() * 1.18)

axes[1].pie(
    fraud_counts.values,
    labels=['Hợp lệ (0)', 'Gian lận (1)'],
    autopct='%1.2f%%',
    colors=colors,
    explode=(0, 0.12),
    startangle=140,
    shadow=True,
    textprops={'fontweight': 'bold'}
)
axes[1].set_title('Tỷ lệ phần trăm lớp 0 vs lớp 1', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.show()

Quantitative Feature Distributions, Skewness, and Mean/Median Comparison

In [ ]:
print('=' * 70)
print('2. PHÂN TÍCH BIẾN ĐỊNH LƯỢNG: PHÂN PHỐI, ĐỘ LỆCH & SO SÁNH LỚP 0 vs 1')
print('=' * 70)

quant_cols = [
    'amount_usd',
    'hours_since_last_txn',
    'txn_count_last_24h',
    'distance_from_home_km',
    'account_balance_usd',
    'velocity_score',
    'merchant_risk_score'
]

# 2.1 Bảng thống kê Skewness, Kurtosis và so sánh Mean / Median theo is_fraud
stats_records = []
for col in quant_cols:
    s_all = df[col].dropna()
    s_0 = df.loc[df['is_fraud'] == 0, col].dropna()
    s_1 = df.loc[df['is_fraud'] == 1, col].dropna()

    stats_records.append({
        'Biến định lượng': col,
        'Skewness (Tổng)': s_all.skew(),
        'Kurtosis (Tổng)': s_all.kurt(),
        'Mean (Fraud=0)': s_0.mean(),
        'Mean (Fraud=1)': s_1.mean(),
        'Chênh lệch Mean (%)': ((s_1.mean() - s_0.mean()) / (abs(s_0.mean()) + 1e-9)) * 100,
        'Median (Fraud=0)': s_0.median(),
        'Median (Fraud=1)': s_1.median(),
        'Chênh lệch Median (%)': ((s_1.median() - s_0.median()) / (abs(s_0.median()) + 1e-9)) * 100
    })

quant_stats_df = pd.DataFrame(stats_records).round(3)
display(quant_stats_df)

# 2.2 Vẽ biểu đồ KDE + Boxplot cho từng biến định lượng theo is_fraud
fig, axes = plt.subplots(len(quant_cols), 2, figsize=(16, 4 * len(quant_cols)))

for idx, col in enumerate(quant_cols):
    skew_val = df[col].skew()

    # Cột trái: Histogram + KDE
    sns.histplot(
        data=df, x=col, hue='is_fraud', kde=True, stat='density',
        common_norm=False, palette={0: '#2b5c8f', 1: '#d9534f'},
        bins=40, alpha=0.4, ax=axes[idx, 0]
    )
    axes[idx, 0].set_title(f'Phân phối mật độ: {col} (Skewness = {skew_val:.2f})', fontweight='bold')
    axes[idx, 0].set_xlabel(col)
    axes[idx, 0].set_ylabel('Mật độ (Density)')

    # Cột phải: Boxplot ngang so sánh lớp 0 và lớp 1
    sns.boxplot(
        data=df, x=col, y=df['is_fraud'].astype(str),
        order=['0', '1'], palette={'0': '#2b5c8f', '1': '#d9534f'},
        ax=axes[idx, 1], flierprops={'marker': 'o', 'markersize': 2, 'alpha': 0.3}
    )
    axes[idx, 1].set_title(f'Boxplot so sánh {col} theo is_fraud', fontweight='bold')
    axes[idx, 1].set_ylabel('is_fraud')
    axes[idx, 1].set_xlabel(col)

plt.tight_layout()
plt.show()

Fraud Rate by Time, Categorical Features & Binary Flag Risk Multiplier

Fraud Rate Over Time (`time_of_day_hour`, `day_of_week`)

In [ ]:
print('=' * 70)
print('3A. PHÂN TÍCH TỶ LỆ GIAN LẬN THEO THỜI GIAN')
print('=' * 70)

base_fraud_rate = df['is_fraud'].mean() * 100
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# 1. Theo giờ trong ngày (time_of_day_hour)
hour_stats = df.groupby('time_of_day_hour')['is_fraud'].agg(
    total_txn='count',
    fraud_count='sum',
    fraud_rate='mean'
).reset_index()
hour_stats['fraud_rate_pct'] = hour_stats['fraud_rate'] * 100

ax1 = axes[0]
ax1_sub = ax1.twinx()
ax1.bar(hour_stats['time_of_day_hour'], hour_stats['total_txn'], color='#b0c4de', alpha=0.5, label='Tổng số giao dịch')
ax1_sub.plot(hour_stats['time_of_day_hour'], hour_stats['fraud_rate_pct'], color='#d9534f', marker='o', linewidth=2.5, label='Tỷ lệ gian lận (%)')
ax1_sub.axhline(base_fraud_rate, color='black', linestyle='--', linewidth=1.2, label=f'TB toàn tập ({base_fraud_rate:.2f}%)')

ax1.set_title('Khối lượng giao dịch & Tỷ lệ gian lận theo Giờ trong ngày', fontweight='bold')
ax1.set_xlabel('Giờ trong ngày (time_of_day_hour)')
ax1.set_ylabel('Số lượng giao dịch', color='#2b5c8f')
ax1_sub.set_ylabel('Tỷ lệ gian lận (%)', color='#d9534f')
ax1.set_xticks(sorted(df['time_of_day_hour'].dropna().unique()))
ax1_sub.legend(loc='upper right')

# 2. Theo ngày trong tuần (day_of_week)
dow_stats = df.groupby('day_of_week')['is_fraud'].agg(
    total_txn='count',
    fraud_count='sum',
    fraud_rate='mean'
).reset_index()
dow_stats['fraud_rate_pct'] = dow_stats['fraud_rate'] * 100

day_order_map = {
    'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3,
    'Friday': 4, 'Saturday': 5, 'Sunday': 6,
    'Mon': 0, 'Tue': 1, 'Wed': 2, 'Thu': 3, 'Fri': 4, 'Sat': 5, 'Sun': 6
}
if dow_stats['day_of_week'].dtype == object and set(dow_stats['day_of_week']).issubset(set(day_order_map.keys())):
    dow_stats['sort_key'] = dow_stats['day_of_week'].map(day_order_map)
    dow_stats = dow_stats.sort_values('sort_key').drop(columns='sort_key')

ax2 = axes[1]
ax2_sub = ax2.twinx()
x_labels = dow_stats['day_of_week'].astype(str)
ax2.bar(x_labels, dow_stats['total_txn'], color='#b0c4de', alpha=0.5, label='Tổng số giao dịch')
ax2_sub.plot(x_labels, dow_stats['fraud_rate_pct'], color='#d9534f', marker='s', linewidth=2.5, label='Tỷ lệ gian lận (%)')
ax2_sub.axhline(base_fraud_rate, color='black', linestyle='--', linewidth=1.2, label=f'TB toàn tập ({base_fraud_rate:.2f}%)')

ax2.set_title('Khối lượng giao dịch & Tỷ lệ gian lận theo Ngày trong tuần', fontweight='bold')
ax2.set_xlabel('Ngày trong tuần (day_of_week)')
ax2.set_ylabel('Số lượng giao dịch', color='#2b5c8f')
ax2_sub.set_ylabel('Tỷ lệ gian lận (%)', color='#d9534f')
ax2_sub.legend(loc='upper right')

plt.tight_layout()
plt.show()

Fraud Rate by Categorical Features (`merchant_category`, `card_type`, `auth_method`, `channel`, `device_type`)

In [ ]:
print('=' * 70)
print('3B. PHÂN TÍCH TỶ LỆ GIAN LẬN THEO NHÓM BIẾN PHÂN LOẠI')
print('=' * 70)

cat_cols = ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']
fig, axes = plt.subplots(3, 2, figsize=(16, 15))
axes = axes.flatten()

for idx, col in enumerate(cat_cols):
    cat_summary = df.groupby(col)['is_fraud'].agg(
        So_luong_GD='count',
        So_GD_Gian_lan='sum',
        Ty_le_Gian_lan_Pct=lambda x: x.mean() * 100
    ).sort_values(by='Ty_le_Gian_lan_Pct', ascending=False).reset_index()

    print(f'\n--- Thống kê theo {col} ---')
    display(cat_summary.round(3))

    sns.barplot(
        data=cat_summary, x='Ty_le_Gian_lan_Pct', y=col,
        palette='Reds_r', edgecolor='black', ax=axes[idx]
    )
    axes[idx].axvline(base_fraud_rate, color='navy', linestyle='--', linewidth=1.5, label=f'TB ({base_fraud_rate:.2f}%)')
    axes[idx].set_title(f'Tỷ lệ gian lận (%) theo {col}', fontweight='bold')
    axes[idx].set_xlabel('Tỷ lệ gian lận (%)')
    axes[idx].set_ylabel(col)
    axes[idx].legend(loc='lower right')

    for p in axes[idx].patches:
        width = p.get_width()
        axes[idx].text(width + 0.1, p.get_y() + p.get_height() / 2, f'{width:.2f}%',
                       va='center', fontsize=10, fontweight='bold')

axes[-1].axis('off')
plt.tight_layout()
plt.show()

Risk Multiplier for the 6 Binary Flags

In [ ]:
print('=' * 70)
print('3C. HỆ SỐ NHÂN RỦI RO (RISK MULTIPLIER) CỦA 6 CỜ NHỊ PHÂN')
print('=' * 70)

binary_flags = [
    'is_foreign_transaction',
    'is_new_merchant',
    'used_vpn',
    'ip_country_mismatch',
    'billing_shipping_mismatch',
    'is_ai_generated_scam_attempt'
]

overall_fraud_rate = df['is_fraud'].mean()
risk_records = []

for flag in binary_flags:
    rate_when_1 = df.loc[df[flag] == 1, 'is_fraud'].mean()
    rate_when_0 = df.loc[df[flag] == 0, 'is_fraud'].mean()
    count_when_1 = (df[flag] == 1).sum()

    risk_multiplier = rate_when_1 / rate_when_0 if rate_when_0 > 0 else np.nan
    baseline_lift = rate_when_1 / overall_fraud_rate if overall_fraud_rate > 0 else np.nan

    risk_records.append({
        'Cờ nhị phân (Binary Flag)': flag,
        'Số GD có Flag=1': count_when_1,
        'Tỷ lệ xuất hiện Flag=1 (%)': (count_when_1 / len(df)) * 100,
        'Fraud Rate khi Flag=0 (%)': rate_when_0 * 100,
        'Fraud Rate khi Flag=1 (%)': rate_when_1 * 100,
        'Risk Multiplier (Flag=1 / Flag=0)': risk_multiplier,
        'Lift so với TB toàn tập': baseline_lift
    })

risk_df = pd.DataFrame(risk_records).sort_values(by='Risk Multiplier (Flag=1 / Flag=0)', ascending=False).round(3)
display(risk_df)

plt.figure(figsize=(10, 5))
ax = sns.barplot(
    data=risk_df,
    x='Risk Multiplier (Flag=1 / Flag=0)',
    y='Cờ nhị phân (Binary Flag)',
    palette='flare',
    edgecolor='black'
)
plt.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='Ngưỡng trung tính (1.0x)')
plt.title('Hệ số nhân rủi ro (Risk Multiplier: Fraud Rate khi Flag=1 chia cho Flag=0)', fontweight='bold', fontsize=13)
plt.xlabel('Risk Multiplier (x lần)')
plt.ylabel('Cờ nhị phân')

for p in ax.patches:
    width = p.get_width()
    ax.text(width + 0.05, p.get_y() + p.get_height() / 2, f'{width:.2f}x',
            va='center', fontweight='bold')

plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 4. Correlation Matrices (Spearman & Pearson) and Top 10 Features Correlated with `is_fraud`

In [ ]:
print('=' * 70)
print('4. MA TRẬN TƯƠNG QUAN (PEARSON & SPEARMAN) VÀ TOP 10 ĐẶC TRƯNG')
print('=' * 70)

numeric_and_binary_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != 'id']

for c in binary_flags:
    if c in df.columns and c not in numeric_and_binary_cols:
        df[c] = df[c].astype(int)
        numeric_and_binary_cols.append(c)

corr_pearson = df[numeric_and_binary_cols].corr(method='pearson')
corr_spearman = df[numeric_and_binary_cols].corr(method='spearman')

# 4.1 Vẽ Heatmap Pearson và Spearman
fig, axes = plt.subplots(1, 2, figsize=(24, 10))
mask = np.triu(np.ones_like(corr_pearson, dtype=bool), k=1)

sns.heatmap(
    corr_pearson, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 8}, ax=axes[0]
)
axes[0].set_title('Ma trận tương quan tuyến tính (Pearson Correlation)', fontweight='bold', fontsize=14)

sns.heatmap(
    corr_spearman, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 8}, ax=axes[1]
)
axes[1].set_title('Ma trận tương quan hạng (Spearman Correlation)', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.show()

# 4.2 Trích xuất Top 10 biến tương quan mạnh nhất với is_fraud (theo giá trị tuyệt đối)
target_corr = pd.DataFrame({
    'Pearson_r': corr_pearson['is_fraud'].drop('is_fraud'),
    'Abs_Pearson': corr_pearson['is_fraud'].drop('is_fraud').abs(),
    'Spearman_rho': corr_spearman['is_fraud'].drop('is_fraud'),
    'Abs_Spearman': corr_spearman['is_fraud'].drop('is_fraud').abs()
})

top10_pearson = target_corr.sort_values(by='Abs_Pearson', ascending=False).head(10)
top10_spearman = target_corr.sort_values(by='Abs_Spearman', ascending=False).head(10)

print('\n>>> TOP 10 ĐẶC TRƯNG TƯƠNG QUAN MẠNH NHẤT VỚI is_fraud THEO PEARSON:')
display(top10_pearson[['Pearson_r', 'Abs_Pearson', 'Spearman_rho']].round(4))

print('\n>>> TOP 10 ĐẶC TRƯNG TƯƠNG QUAN MẠNH NHẤT VỚI is_fraud THEO SPEARMAN:')
display(top10_spearman[['Spearman_rho', 'Abs_Spearman', 'Pearson_r']].round(4))

# 4.3 Biểu đồ so sánh Top 10 biến tương quan với is_fraud
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

colors_p = ['#d9534f' if v > 0 else '#2b5c8f' for v in top10_pearson['Pearson_r']]
axes[0].barh(top10_pearson.index[::-1], top10_pearson['Pearson_r'][::-1], color=colors_p[::-1], edgecolor='black')
axes[0].axvline(0, color='black', linewidth=1)
axes[0].set_title('Top 10 biến tương quan Pearson mạnh nhất với is_fraud', fontweight='bold')
axes[0].set_xlabel('Hệ số tương quan Pearson (r)')

colors_s = ['#d9534f' if v > 0 else '#2b5c8f' for v in top10_spearman['Spearman_rho']]
axes[1].barh(top10_spearman.index[::-1], top10_spearman['Spearman_rho'][::-1], color=colors_s[::-1], edgecolor='black')
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('Top 10 biến tương quan Spearman mạnh nhất với is_fraud', fontweight='bold')
axes[1].set_xlabel('Hệ số tương quan Spearman (ρ)')

plt.tight_layout()
plt.show()